[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/29_cnn/04_cnn_on_real_images.ipynb)

# Bài 04 — CNN trên ảnh thật: Fashion-MNIST và CIFAR-10

**🎯 Mục tiêu**
- Huấn luyện CNN trên hai bộ ảnh thật và so sánh **trên cùng dữ liệu** với những mô hình đã học: tuyến tính (chương 07) và MLP (chương 27).
- Quy trình chuẩn cho ảnh màu: chuẩn hoá theo kênh, data augmentation (lật ngang, dịch chuyển), Batch Normalization, lịch learning rate *one-cycle*.
- Hiểu vì sao CNN vượt xa "trần ~40%" của mô hình tuyến tính trên CIFAR-10 — và những gì vẫn còn khó.

> Dùng lại dữ liệu đã tải ở chương 07 (trong `data/raw/`; nếu chưa có, lần đầu tải ~200 MB). Có GPU thì bài chạy trong vài chục giây; chỉ CPU thì mất vài phút.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/29_cnn

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.real_datasets import CIFAR10_CLASSES, FASHION_MNIST_CLASSES, load_cifar10, load_fashion_mnist
from ml_course.visualization import plot_confusion_matrix, set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"PyTorch {torch.__version__} | thiết bị: {device}")

Một hàm huấn luyện dùng chung cho cả bài: Adam + lịch **one-cycle** (learning rate tăng dần rồi giảm về rất nhỏ ở cuối — kết quả cuối ổn định, không dao động như learning rate cố định; xem chương 28), có tuỳ chọn augmentation, ghi accuracy **validation** sau mỗi epoch.

In [ ]:
def predict(model, X, batch_size=1000):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[s:s + batch_size].to(device)).argmax(1).cpu() for s in range(0, len(X), batch_size)])


def accuracy(model, X, y):
    return float((predict(model, X) == y).float().mean())


def train_cnn(model, X_tr, y_tr, X_va, y_va, epochs, lr=1e-3, batch_size=128, augment=None, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    steps = epochs * ((len(X_tr) + batch_size - 1) // batch_size)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3 * lr, total_steps=steps)
    history, t0 = [], time.perf_counter()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr), generator=g)
        for s in range(0, len(perm), batch_size):
            idx = perm[s:s + batch_size]
            xb = X_tr[idx] if augment is None else augment(X_tr[idx], g)
            optimizer.zero_grad()
            nn.functional.cross_entropy(model(xb.to(device)), y_tr[idx].to(device)).backward()
            optimizer.step()
            scheduler.step()
        history.append(accuracy(model, X_va, y_va))
        print(f"  epoch {epoch + 1}: accuracy validation = {history[-1]:.4f} ({time.perf_counter() - t0:.0f}s)")
    return history

## 1. Fashion-MNIST: tuyến tính → MLP → CNN

Giữ 5 000 ảnh train làm validation, 10 000 ảnh test chỉ dùng ở cuối. Chuẩn hoá pixel về trung bình 0, độ lệch chuẩn 1 (thống kê tính trên phần train).

In [ ]:
Xf_train, yf_train, Xf_test, yf_test = load_fashion_mnist()
perm = np.random.default_rng(0).permutation(len(yf_train))
tr, va = perm[5000:], perm[:5000]
mu, sigma = Xf_train[tr].mean() / 255, Xf_train[tr].std() / 255


def to_fashion_tensor(images):
    return ((torch.tensor(images, dtype=torch.float32) / 255 - mu) / sigma)[:, None]   # (N, 1, 28, 28)


Xf_tr, Xf_va, Xf_te = to_fashion_tensor(Xf_train[tr]), to_fashion_tensor(Xf_train[va]), to_fashion_tensor(Xf_test)
yf_tr, yf_va, yf_te = torch.tensor(yf_train[tr]), torch.tensor(yf_train[va]), torch.tensor(yf_test)
print("train", tuple(Xf_tr.shape), "| validation", tuple(Xf_va.shape), "| test", tuple(Xf_te.shape))

fashion_cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 28 → 14
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 14 → 7
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10),
)
train_cnn(fashion_cnn, Xf_tr, yf_tr, Xf_va, yf_va, epochs=3)
fashion_test_acc = accuracy(fashion_cnn, Xf_te, yf_te)
print(f"CNN — accuracy TEST = {fashion_test_acc:.4f}")

| Mô hình trên Fashion-MNIST | Bài | Accuracy test |
|---|---|---|
| Softmax regression (tuyến tính) | [chương 07 · assignment 93](../07_logistic_regression/93_assignment_softmax_regression.ipynb) | ~83% |
| MLP 2 lớp ẩn (Keras) | [chương 27 · bài 05](../27_deep_learning_foundations/05_tensorflow_and_keras.ipynb) | ~88% |
| CNN 2 lớp tích chập (bài này) | — | ~91% |

Mỗi bước thêm một loại **thiên kiến quy nạp** phù hợp với ảnh: lớp ẩn → đặc trưng phi tuyến; tích chập → đặc trưng **cục bộ**, dùng chung trọng số ở mọi vị trí.

## 2. CIFAR-10: vượt "trần" của mô hình tuyến tính

[Chương 07 · bài 08](../07_logistic_regression/08_linear_classifiers_cifar10.ipynb) cho thấy mô hình tuyến tính trên pixel thô dừng ở **~40%**, và learning curve cho biết thêm dữ liệu không giúp được bao nhiêu. Dùng **đúng 20 000 ảnh train và 2 000 ảnh validation** như bài đó:

In [ ]:
Xc_train_all, yc_train_all, Xc_test, yc_test = load_cifar10()
idx = np.random.default_rng(42).permutation(len(yc_train_all))       # cùng cách chia với chương 07 · bài 08
train_idx, val_idx = idx[:20_000], idx[20_000:22_000]
channel_mean = Xc_train_all[train_idx].reshape(-1, 3).mean(axis=0) / 255
channel_std = Xc_train_all[train_idx].reshape(-1, 3).std(axis=0) / 255
print("trung bình theo kênh (R, G, B):", channel_mean.round(3), "| độ lệch chuẩn:", channel_std.round(3))


def to_cifar_tensor(images):
    x = (torch.tensor(images, dtype=torch.float32) / 255 - torch.tensor(channel_mean, dtype=torch.float32)) / torch.tensor(channel_std, dtype=torch.float32)
    return x.permute(0, 3, 1, 2).contiguous()                         # (N, 32, 32, 3) → (N, 3, 32, 32)


Xc_tr, Xc_va, Xc_te = to_cifar_tensor(Xc_train_all[train_idx]), to_cifar_tensor(Xc_train_all[val_idx]), to_cifar_tensor(Xc_test)
yc_tr, yc_va, yc_te = torch.tensor(yc_train_all[train_idx]), torch.tensor(yc_train_all[val_idx]), torch.tensor(yc_test)
del Xc_train_all
print("train", tuple(Xc_tr.shape), "| validation", tuple(Xc_va.shape), "| test", tuple(Xc_te.shape))

**Data augmentation** cho ảnh tự nhiên: **lật ngang** (một con ngựa quay trái hay phải vẫn là ngựa — nhưng không lật dọc: tàu thuỷ lộn ngược không phải ảnh tự nhiên) và **dịch chuyển** ngẫu nhiên tối đa 4 pixel (đệm phản chiếu rồi cắt lại 32×32). Mỗi epoch mô hình thấy một phiên bản hơi khác của mỗi ảnh:

In [ ]:
def flip_and_shift(xb, g, max_shift=4):
    flip = torch.rand(len(xb), generator=g) < 0.5
    xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
    padded = nn.functional.pad(xb, (max_shift,) * 4, mode="reflect")
    i, j = torch.randint(0, 2 * max_shift + 1, (2,), generator=g).tolist()
    return padded[:, :, i:i + 32, j:j + 32]


demo = flip_and_shift(Xc_tr[:8], torch.Generator().manual_seed(1))
fig, axes = plt.subplots(2, 8, figsize=(12, 3.3))
for k in range(8):
    for row, img in enumerate([Xc_tr[k], demo[k]]):
        shown = img.permute(1, 2, 0).numpy() * channel_std + channel_mean
        axes[row, k].imshow(shown.clip(0, 1))
        axes[row, k].axis("off")
axes[0, 0].set_title("gốc", loc="left")
axes[1, 0].set_title("sau augmentation", loc="left")
plt.show()

Kiến trúc: 3 khối "Conv–BN–ReLU ×2 + MaxPool" (16 → 32 → 64 kênh), rồi **global average pooling** và một lớp tuyến tính (chương 29 · bài 02–03):

In [ ]:
def conv_block(c_in, c_out):
    return [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.MaxPool2d(2)]


cifar_cnn = nn.Sequential(*conv_block(3, 16), *conv_block(16, 32), *conv_block(32, 64),   # 32 → 16 → 8 → 4
                          nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10))
n_params = sum(p.numel() for p in cifar_cnn.parameters())
print(f"CNN: {n_params:,} tham số | mô hình tuyến tính ở chương 07: {3073 * 10:,} tham số")
cifar_history = train_cnn(cifar_cnn, Xc_tr, yc_tr, Xc_va, yc_va, epochs=8, augment=flip_and_shift)
cifar_test_acc = accuracy(cifar_cnn, Xc_te, yc_te)
print(f"CNN — accuracy TEST = {cifar_test_acc:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(range(1, len(cifar_history) + 1), cifar_history, "o-", label="CNN — validation")
ax.axhline(0.40, color="C3", ls="--", label="mô hình tuyến tính (chương 07 · bài 08)")
ax.axhline(0.10, color="gray", ls=":", label="đoán ngẫu nhiên")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("CIFAR-10, cùng 20 000 ảnh train")
plt.show()

Với chỉ khoảng **2.4 lần** số tham số của mô hình tuyến tính, CNN đạt khoảng **73%** — gần gấp đôi. Khác biệt không nằm ở số tham số mà ở **cấu trúc**: bộ lọc nhỏ dò đặc trưng cục bộ (cạnh, màu, kết cấu), dùng chung ở mọi vị trí; pooling cho phép nhận ra vật thể dù nó nằm ở đâu; các lớp chồng lên nhau ghép đặc trưng đơn giản thành bộ phận vật thể. Mô hình tuyến tính chỉ có **một ảnh mẫu** cho mỗi lớp.

## 3. Còn khó ở đâu?

In [ ]:
pred_te = predict(cifar_cnn, Xc_te).numpy()
cm = np.zeros((10, 10), int)
np.add.at(cm, (yc_test, pred_te), 1)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
plot_confusion_matrix(cm, labels=CIFAR10_CLASSES, ax=ax, normalize=True, title="CNN trên CIFAR-10 (test, chuẩn hoá theo hàng)")
plt.xticks(rotation=45, ha="right")
plt.show()
off = cm - np.diag(np.diag(cm))
for k in np.argsort(off.ravel())[::-1][:3]:
    i, j = divmod(k, 10)
    print(f"{CIFAR10_CLASSES[i]:>9s} bị đoán là {CIFAR10_CLASSES[j]:<9s}: {off[i, j]} ảnh")
per_class = cm.diagonal() / cm.sum(axis=1)
print("lớp khó nhất:", CIFAR10_CLASSES[int(per_class.argmin())], f"({per_class.min():.2f})",
      "| dễ nhất:", CIFAR10_CLASSES[int(per_class.argmax())], f"({per_class.max():.2f})")

Vài ảnh test bị đoán sai — nhiều ảnh khó cả với người (32×32 pixel rất nhỏ):

In [ ]:
wrong = np.flatnonzero(pred_te != yc_test)[:12]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.6))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(Xc_test[i])
    ax.set_title(f"thật: {CIFAR10_CLASSES[yc_test[i]]}\nđoán: {CIFAR10_CLASSES[pred_te[i]]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Muốn đi xa hơn** (tham khảo, không chạy trong bài): dùng đủ 50 000 ảnh, mạng sâu hơn với kết nối tắt (ResNet — [bài 03](03_architectures_lenet_to_resnet.ipynb)), huấn luyện vài chục epoch với augmentation mạnh hơn → khoảng 90–95%; hoặc **transfer learning** từ mô hình đã huấn luyện trên ImageNet ([chương 33](../33_computer_vision/README.md)).

## 🧠 Kiểm tra nhanh

1. Vì sao lật **ngang** là augmentation hợp lệ cho CIFAR-10 còn lật **dọc** thì không? Với ảnh chữ số thì sao?
2. Thống kê chuẩn hoá theo kênh được tính trên tập nào? Vì sao?
3. CNN ở bài này có số tham số chỉ cỡ 2.4 lần mô hình tuyến tính nhưng accuracy gần gấp đôi. Giải thích.
4. Learning rate *one-cycle* giúp gì so với learning rate cố định ở cuối quá trình huấn luyện?
5. Các cặp lớp bị nhầm nhiều nhất của CNN có giống với của mô hình tuyến tính (chương 07 · bài 08) không? Điều đó nói gì?

## 📌 Tóm tắt

- Cùng dữ liệu, thêm đúng thiên kiến quy nạp cho ảnh: Fashion-MNIST ~83% (tuyến tính) → ~88% (MLP) → ~91% (CNN); CIFAR-10 ~40% (tuyến tính) → ~73% (CNN nhỏ, 20 000 ảnh, 8 epoch).
- Quy trình ảnh màu: chuẩn hoá theo kênh (thống kê từ train), augmentation hợp lý với miền dữ liệu, BatchNorm, lịch learning rate.
- Đánh giá test **một lần** ở cuối; theo dõi quá trình bằng validation.

## 📚 Tài liệu tham khảo

- A. Krizhevsky (2009) — *Learning Multiple Layers of Features from Tiny Images*; H. Xiao, K. Rasul, R. Vollgraf (2017) — *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*.
- L. N. Smith, N. Topin (2017) — *Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates* (one-cycle).
- CS231n — [Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/).